# Fleet Force — Motor Analítico Unificado

**Uma fonte de verdade:** `fleet_force_engine` (Python modular). Este notebook **não duplica os cálculos**; executa a mesma API utilizada pelo CLI e pelo backend.

**Prioridade:** regras auditáveis (UFMG + ANP + PBEV). **ML:** somente se superar baseline e validar independentemente. Dados de 2025 não comprovam causalidade nem economia realizada.

**Importante:** execute dentro da pasta do projeto extraída do ZIP. Instale as dependências do arquivo `requirements.txt` se necessário.

## 1. Ambiente, arquivos e configuração

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT=Path.cwd().resolve()
if not (ROOT/'fleet_force_engine').exists():
    ROOT=ROOT.parent
if not (ROOT/'fleet_force_engine').exists():
    raise FileNotFoundError('Extraia o pacote unificado e execute este notebook na pasta notebooks/ ou na raiz do pacote.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))

from fleet_force_engine import Settings,run_engine
pd.set_option('display.max_columns',80)
print('Pasta do motor:',ROOT)

In [ ]:
# Ajuste os caminhos se executar no Colab ou em outra pasta.
def first_present(*candidates):
    return next((str(Path(p)) for p in candidates if p and Path(p).is_file()),None)

FLEET_CSV=first_present(ROOT/'fleet_analysis_outputs/fleet_vehicles.csv')
FLEET_PDF=first_present(ROOT/'data/despesa-veiculos-ufmg-2025.pdf',
                        ROOT/'despesa-veiculos-ufmg-2025.pdf',
                        ROOT/'despesa-veiculos-ufmg-2025(1).pdf',
                        '/mnt/data/despesa-veiculos-ufmg-2025(1).pdf')
PBEV=first_present(ROOT/'data/pbev_catalogo_historico_extraido_auditar.csv')
ANP=first_present(ROOT/'data/anp_precos_mensais_uf.csv')
SOURCE_ZIP=first_present(ROOT/'fleet-force.zip')
EXTERNAL_VALIDATION=None  # Ex.: ROOT/'frota_2026_validada.csv'
UF='MG'  # Só escolha com base na operação. UFMG não estabelece automaticamente UF do abastecimento.
OUTPUT_DIR=ROOT/'outputs'
print({'frota_csv':FLEET_CSV,'frota_pdf':FLEET_PDF,'pbev':PBEV,'anp':ANP,'zip':SOURCE_ZIP})

## 2. Executar um único motor (determinístico + gate ML)

In [ ]:
if not (FLEET_CSV or FLEET_PDF):
    raise FileNotFoundError('Coloque o PDF UFMG na raiz do projeto ou forneça o CSV da frota.')
result=run_engine(
    fleet_csv=FLEET_CSV,
    fleet_pdf=None if FLEET_CSV else FLEET_PDF,
    pbev_csv=PBEV,
    anp_monthly_csv=ANP,
    source_zip=SOURCE_ZIP,
    external_validation_csv=EXTERNAL_VALIDATION,
    cfg=Settings(price_uf=UF),
    output_dir=OUTPUT_DIR,
)
print('Resultado exportado em:',OUTPUT_DIR)

## 3. Resumo financeiro e cobertura

In [ ]:
summary=pd.DataFrame([{'indicador':k,'valor':v} for k,v in result.kpis.items()])
display(summary)
print('Fonte dos litros:',result.metadata['liters_source'])
print('Fonte dos preços:',result.metadata['price_source'])

## 4. Frota por categoria

In [ ]:
display(result.categories.sort_values('total_expense_brl',ascending=False).round(2))
ax=result.categories.sort_values('total_expense_brl').plot.barh(
    x='vehicle_category',y='total_expense_brl',figsize=(10,5),legend=False,title='Despesas registradas por categoria')
ax.set_xlabel('R$ - valores observados');ax.set_ylabel('Categoria');plt.tight_layout();plt.show()

## 5. Benchmarks determinísticos e prioridades

In [ ]:
columns=['fleet_id','model','vehicle_category','km','kmpl',
         'peer_median_kmpl','peer_overuse_pct','unified_priority_liters_est',
         'unified_priority_source','peer_confidence']
ranked=result.fleet.loc[result.fleet.unified_priority_alert,columns].sort_values('unified_priority_liters_est',ascending=False)
display(ranked.head(25).round(3))
print('Os litros são ESTIMADOS; oportunidades de investigação, não economias confirmadas.')

## 6. Despesas por veículo e Pareto

In [ ]:
cols=['fleet_id','model','vehicle_category','total_expense','cost_per_km',
      'fuel_expense','maintenance_expense_known','cost_cumulative_share_pct','pareto_80_flag']
display(result.fleet[cols].sort_values('total_expense',ascending=False).head(20).round(2))
ax=result.fleet['cost_cumulative_share_pct'].dropna().reset_index(drop=True).plot(figsize=(10,4),title='Pareto — participação acumulada de despesa')
ax.axhline(80,linestyle='--');ax.set_xlabel('Veículos ordenados por despesa');ax.set_ylabel('Despesa acumulada (%)');plt.show()

## 7. Gate automático: modelos aprovados ou rejeitados

In [ ]:
for kind,metrics in result.training_metrics.items():
    print('MODELO:',kind.upper())
    display(pd.DataFrame([{'critério':k,'valor':v} for k,v in metrics.items()
                          if k not in {'trained_fleet_ids','segment_metrics','external_holdout'}]))
    if metrics.get('segment_metrics'):
        display(pd.DataFrame(metrics['segment_metrics']).round(4))
print('Referências com ML validado efetivamente usadas:',
      int(result.fleet.active_reference_source.eq('ml_approved_external_validation').sum()))

## 8. Previsões de consumo fora da amostra — **exploratórias**

In [ ]:
if result.crossval_predictions.empty:
    print('Sem amostra suficiente para backtest.')
else:
    display(result.crossval_predictions.sort_values('ml_oof_overuse_pct',ascending=False).head(15).round(2))
print('Estas previsões são cross-validation sobre 2025. Sem validação independente não geram alertas.')

## 9. ANP e cenários financeiros independentes

In [ ]:
display(result.prices.round(3))
scenarios=pd.read_csv(OUTPUT_DIR/'fleet_scenarios.csv')
display(scenarios.round(2))
print('Cenários por combustível são ALTERNATIVOS. Não somar gasolina, etanol e diesel.')
print('Previsão ANP de um mês aprovada?',result.training_metrics['price'].get('approved_production'))

## 10. Referência PBEV: associação exploratória, sem inferência de versão

In [ ]:
fields=['fleet_id','model','pbev_min_kmpl','pbev_reference_year',
        'pbev_excess_pct','pbev_match_method','pbev_comparison_confidence']
linked=result.fleet.loc[result.fleet.pbev_min_kmpl.notna(),fields]
display(linked.head(25).round(3))
print('Associação não comprova motor/ano/combustível equivalentes; nunca substituir benchmark confiável por nome aproximado.')

## 11. Anomalias para auditoria (não são diagnósticos)

In [ ]:
anomalies=result.fleet.loc[result.fleet.anomaly_flag_ml,
    ['fleet_id','model','vehicle_category','km','kmpl','total_expense',
     'cost_per_km','anomaly_score_ml','anomaly_status_ml']]
display(anomalies.sort_values('anomaly_score_ml',ascending=False).head(25).round(2))

## 12. Consulta individual por veículo

In [ ]:
VEHICLE_ID=result.fleet.iloc[0].fleet_id
found=result.fleet.loc[result.fleet.fleet_id.eq(VEHICLE_ID)]
if found.empty:raise ValueError(f'Veículo não encontrado: {VEHICLE_ID}')
fields=['fleet_id','model','vehicle_category','km','kmpl','total_expense','cost_per_km',
        'fuel_expense','maintenance_expense_known','quality_status','quality_reason',
        'peer_median_kmpl','peer_confidence','active_reference_source','active_reference_kmpl',
        'unified_priority_alert','unified_priority_liters_est','anomaly_status_ml']
display(found[fields].T.rename(columns={found.index[0]:'valor'}))

## 13. Artefatos prontos para API e Angular

In [ ]:
print('Arquivos exportados:')
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():print(f'  {p.name} ({p.stat().st_size:,} bytes)')
print('Instale requirements-api.txt e execute uvicorn api:app --port 8000 para os endpoints REST.')